# 什么是 Pydantic？

Pydantic 是 Python 中最常用的数据校验与配置管理库，它利用类型注解来保证数据结构的完整性。它能在运行时校验、解析并强制转换数据，使其符合声明的类型，因此非常适合构建稳健的 API、处理外部输入。其核心校验逻辑用 Rust 编写，性能很高。

# 核心特性与优势

**核心特性与优势**

* **数据校验与解析：** 用标准 Python 类型定义数据结构，并自动强制执行这些规则。
* **类型注解集成：** 用 Python 类型注解定义 schema，减少冗长的手写校验代码。
* **高性能：** 核心校验引擎用 Rust 实现，速度极快。
* **严格模式与宽松模式：** 支持严格模式（强制类型完全匹配）和宽松模式（尝试类型转换，例如把 `"1"` 转成 `1`）。
* **清晰的错误信息：** 校验失败时提供详细错误说明。
* **JSON Schema 生成：** Pydantic 模型可轻松生成 JSON Schema，用于文档或其他语言中的校验。 

# 没有 Pydantic 时的问题

In [13]:
def add_patient_data(name: str, age: int):
    if type(name) == str and type(age) == int:
        if age >= 0:
            print(name)
            print(age)
            print("Data added successfully to the database!")
        else:
            raise ValueError("Age cannot be negative.")
    else:
        raise TypeError("Invalid data type for name or age. Name should be a string and age should be an integer.")
    


def update_patient_data(name: str, age: int):
    if type(name) == str and type(age) == int:
        if age >= 0:
            print(name)
            print(age)
            print("Data updated successfully in the database!")
        else:
            raise ValueError("Age cannot be negative.")
    else:
        raise TypeError("Invalid data type for name or age. Name should be a string and age should be an integer.")

   

In [15]:
add_patient_data("Bappy", 25)

Bappy
25
Data added successfully to the database!


# 如何使用 Pydantic

1. **定义一个 Pydantic 模型**，表示数据的**理想 schema**。
  * 包括期望的字段、类型，以及校验约束（例如用 `gt=0` 表示正数）。


2. **用原始输入数据实例化模型**（通常是字典或类 JSON 结构）。
  * Pydantic 会自动**校验**数据，并在可能时**强制转换**为正确的 Python 类型。
  * 若数据不符合模型要求，Pydantic 会抛出 `ValidationError`。


3. **把校验后的模型对象**传给函数，或在整个代码库中使用。
  * 这样能保证程序各处拿到的都是**干净、类型安全、逻辑合法的数据**。

In [34]:
from pydantic import BaseModel

class PatientData(BaseModel):
    name: str
    age: int
    weight: float


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data added successfully to the database!")


def update_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data updated successfully in the database!")

patient_data = {"name": "Bappy", "age": "25", "weight": "70.5"}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
Data added successfully to the database!


In [32]:
patient_data = {"name": "Alex", "age": "25", "weight": "70.5"}

patient_2 = PatientData(**patient_data)

update_patient_data(patient_2)

Alex
25
70.5
Data updated successfully in the database!


# 稍复杂一点的例子

In [37]:
from pydantic import BaseModel
from typing import List, Dict

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data added successfully to the database!")


def update_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data updated successfully in the database!")

patient_data = {"name": "Bappy", "age": "25", "weight": "70.5", "married": "True", "allergies": ["peanuts", "shellfish"], "contact_info": {"email": "bappy@example.com", "phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
Data added successfully to the database!


# 必填字段与可选字段

In [43]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = None
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "age": "25", "weight": "70.5", "contact_info": {"email": "bappy@example.com", "phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
False
None
Data added successfully to the database!


# 数据校验

In [56]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional

class PatientData(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=100)
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = Field(max_length=5)
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "email": "bappy@gmail.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": "70.5", 
                "allergies": ["peanuts", "shellfish"],
                "contact_info": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
False
['peanuts', 'shellfish']
Data added successfully to the database!


In [60]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: Annotated[str, Field(max_length=50, title='Name of the patient', description='Give the name of the patient in less than 50 chars', examples=['Bappy', 'Alex'])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict= True, description='Weight of the patient in kg')]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: Dict[str, str]



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "email": "bappy@gmail.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": 70.5, 
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
None
['peanuts', 'shellfish']
Data added successfully to the database!


# 字段校验器（Field Validator）

In [66]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):

        valid_domains = ['hdfc.com', 'icici.com']
        # abc@gmail.com
        domain_name = value.split('@')[-1]

        if domain_name not in valid_domains:
            raise ValueError('Not a valid domain')

        return value
    

    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        return value.upper()



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

BAPPY
25
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


# 模型校验器（Model Validator）

In [69]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]


    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            raise ValueError('Patients older than 60 must have an emergency contact')
        return model



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

bappy
70
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


C:\Users\AC\AppData\Local\Temp\ipykernel_6244\3104895771.py:15: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


# 计算字段（Computed Fields）

In [70]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]


    @computed_field
    @property
    def bmi(self) -> float:
        bmi = round(self.weight/(self.height**2),2)
        return bmi



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("BMI:", patient.bmi)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "height": 1.75,
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

bappy
70
70.5
True
['peanuts', 'shellfish']
BMI: 23.02
Data added successfully to the database!


# 嵌套模型（Nested Models）

In [72]:
from pydantic import BaseModel

class Address(BaseModel):

    city: str
    state: str
    pin: str

class PatientData(BaseModel):

    name: str
    gender: str
    age: int
    address: Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}

address1 = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address1}

patient1 = PatientData(**patient_dict)

print(patient1)
print(patient1.name)
print(patient1.address)
print(patient1.address.city)

name='Kishor' gender='male' age=40 address=Address(city='gurgaon', state='haryana', pin='122001')
Kishor
city='gurgaon' state='haryana' pin='122001'
gurgaon


# 序列化（Serialization） 

In [74]:
from pydantic import BaseModel

class Address(BaseModel):

    city: str
    state: str
    pin: str

class PatientData(BaseModel):

    name: str
    gender: str
    age: int
    address: Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}

address1 = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address1}

patient1 = PatientData(**patient_dict)

# temp = patient1.model_dump()
temp = patient1.model_dump_json()

print(temp)
print(type(temp))

{"name":"Kishor","gender":"male","age":40,"address":{"city":"gurgaon","state":"haryana","pin":"122001"}}
<class 'str'>
